# Project 03 · M-Pesa personal finance analytics

**Benson M. Merita · MeritaAnalytics** · Tools: Python (pandas, NumPy, regular expressions), Matplotlib

**Business question.** Where does the money go, is the account holder saving, who gets paid most, which payments look unusual, and what should next quarter's spending look like?

**Data.** A 24-month mobile-money statement (January 2024 to December 2025, 5,085 raw rows) in the layout of an M-Pesa statement export. It is synthetic: every name, merchant and amount was generated by code. The file has two **answer-key** columns (`gen_category`, `gen_anomaly`). The analysis sets them aside at the start and uses them only to score the results.

| Step | What it answers |
|---|---|
| 1 | Load and clean the statement |
| 2 | Categorise every transaction from its text |
| 3 | Income, spending and savings rate |
| 4 | Who gets paid most? (RFM) |
| 5 | Which payments look unusual? |
| 6 | Forecast next quarter's spending |
| 7 | Final checks |

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

BLUE, GREY, DARK, AMBER = "#1F5FD6", "#9CA3AF", "#0B2A4A", "#B45309"
plt.rcParams.update({"figure.figsize": (10, 3.6), "axes.grid": True, "grid.alpha": 0.3})
pd.set_option("display.width", 120)

RAW = "../datasets/03-mpesa-finance/raw/mpesa_statement_raw.csv"

## Step 1 · Load and clean the statement

**What:** read every column as text, then convert dates and amounts explicitly. Remove exact duplicates and set aside rows with no readable date or no amount, each with a reason.
**Why:** exported statements often have commas in amounts, broken dates and repeated rows. Every raw row must be accounted for: **raw = duplicates + rejected + clean**.

In [ ]:
stmt = pd.read_csv(RAW, dtype=str)
answer_key = stmt[["Receipt No", "gen_category", "gen_anomaly"]].drop_duplicates("Receipt No")
answer_key["gen_anomaly"] = answer_key["gen_anomaly"].astype(int)

s = stmt.drop(columns=["gen_category", "gen_anomaly"]).drop_duplicates()
n_dups = len(stmt) - len(s)
s["Details"] = s["Details"].str.strip().str.replace(r"\s+", " ", regex=True)
s["time"] = pd.to_datetime(s["Completion Time"], format="%Y-%m-%d %H:%M:%S", errors="coerce")
for col in ["Paid In", "Withdrawn", "Balance"]:
    s[col] = pd.to_numeric(s[col].str.replace(",", "", regex=False), errors="coerce")
s["paid_in"], s["withdrawn"] = s["Paid In"].fillna(0.0), s["Withdrawn"].fillna(0.0)

reason = np.select([s["time"].isna(), (s["paid_in"] == 0) & (s["withdrawn"] == 0)],
                   ["unreadable or missing time", "no amount"], default="")
rejects = s[reason != ""].assign(reason=reason[reason != ""])
s = s[reason == ""].sort_values("time", kind="stable").reset_index(drop=True)

print(f"raw {len(stmt)} = duplicates {n_dups} + rejected {len(rejects)} + clean {len(s)} ->",
      len(stmt) == n_dups + len(rejects) + len(s))
print("rejected by reason:", rejects["reason"].value_counts().to_dict())

**Balance check.** Each row's balance should move by exactly that row's amount. Gaps appear only where a rejected row sat between two clean rows.

In [ ]:
step = s["Balance"].diff() - (s["paid_in"] - s["withdrawn"])
print("balance steps that differ from the row amount:", int((step.abs() > 0.01).sum()), "of", int(step.notna().sum()))

## Step 2 · Categorise every transaction

**What:** assign a category from keywords in the `Details` text. Rules are checked **in order**, and the first match wins.
**Why:** a naive rule list makes real mistakes. Below, "Sacco" appears both in savings transfers and in a matatu Sacco's name, so a short rule list files bus fares as savings.

In [ ]:
def categorise(details, rules, default="Other"):
    text = details.str.lower()
    conditions = [text.str.contains(pattern, regex=True) for _, pattern in rules]
    return pd.Series(np.select(conditions, [name for name, _ in rules], default=default), index=details.index)

naive_rules = [("Savings", r"sacco"), ("Cash withdrawal", r"withdraw"), ("Transport", r"matatu|boda|fuel|taxi")]
test = s.merge(answer_key, on="Receipt No", how="left")
test["naive"] = categorise(test["Details"], naive_rules)
wrong = test[(test["naive"] != "Other") & (test["naive"] != test["gen_category"])]
print("rows the naive rules put in the wrong category:", len(wrong))
wrong.groupby(["gen_category", "naive"]).size().rename("rows")

The full rule list puts specific patterns first (for example "transfer to savings" before any transport words):

In [ ]:
RULES = [
    ("Income - salary", r"business payment from"), ("Income - sales", r"customer payment from"),
    ("Income - received", r"funds received from"), ("Income - deposit", r"deposit of funds"),
    ("Rent", r"\brent\b"), ("Savings", r"transfer to savings"), ("Fees", r"transaction charge"),
    ("Cash withdrawal", r"withdraw cash"), ("Airtime and data", r"airtime|data bundles"),
    ("Utilities", r"electricity|fiberhome|internet"), ("Transport", r"matatu|boda|fuel station|taxi"),
    ("Groceries", r"supermarket|grocers|butchery"), ("Eating out", r"hotel|kiosk|eatery"),
    ("Entertainment", r"streambox|betting|cinema"), ("Health", r"pharmacy|clinic"),
    ("Send money", r"customer transfer to"),
]
s["category"] = categorise(s["Details"], RULES)
chk = s.merge(answer_key, on="Receipt No", how="left")
print(f"{len(RULES)} rules | rows left in Other: {int((s['category'] == 'Other').sum())} | "
      f"agreement with the answer key: {100 * (chk['category'] == chk['gen_category']).mean():.1f}%")

## Step 3 · Income, spending and savings rate

**What:** group each category into income, saving or spending, then total by month. Savings rate = (income − spending) ÷ income.
**Why:** this is the headline a person, a lender or a financial coach would ask for first.

In [ ]:
s["amount"] = s["paid_in"] + s["withdrawn"]
s["kind"] = np.where(s["category"].str.startswith("Income"), "income",
                     np.where(s["category"] == "Savings", "saving", "spending"))
s["month"] = s["time"].dt.to_period("M")

flow = s.groupby(["month", "kind"])["amount"].sum().unstack(fill_value=0)
flow["kept"] = flow["income"] - flow["spending"]
tot = flow[["income", "spending", "saving"]].sum()
rate = 100 * (tot["income"] - tot["spending"]) / tot["income"]
print(f"24 months: income Ksh {tot['income']:,.0f} | spending Ksh {tot['spending']:,.0f} | savings rate {rate:.1f}%")
print("months where spending exceeded income:", int((flow["kept"] < 0).sum()), "of", len(flow))

mix = s[s["kind"] == "spending"].groupby("category")["amount"].agg(rows="count", total="sum")
mix["share_pct"] = (100 * mix["total"] / mix["total"].sum()).round(1)
mix = mix.sort_values("total", ascending=False)
mix.head(7).round(0)

In [ ]:
fig, ax = plt.subplots(1, 2, gridspec_kw={"width_ratios": [1.3, 1]})
x = flow.index.to_timestamp()
ax[0].plot(x, flow["income"] / 1000, color=BLUE, lw=2.2, label="Income")
ax[0].plot(x, flow["spending"] / 1000, color=AMBER, lw=2.2, label="Spending")
ax[0].set_title("Monthly income and spending"); ax[0].set_ylabel("Ksh thousand"); ax[0].legend()
top = mix["total"].head(7) / 1e6
ax[1].barh(top.index[::-1], top.values[::-1], color=[AMBER if n == top.index[0] else BLUE for n in top.index[::-1]])
ax[1].set_title("Top spending categories, 24 months"); ax[1].set_xlabel("Ksh million")
plt.tight_layout(); plt.show()

## Step 4 · Who gets paid most? (RFM)

**What:** pull the payee name out of the text, then score each payee from 1 to 4 on **R**ecency (how recently paid), **F**requency (how often) and **M**onetary value (how much in total). Scores are combined into plain-language groups.
**Why:** RFM is a standard way to see which relationships carry most of the money.

In [ ]:
skip = ["Fees", "Airtime and data", "Cash withdrawal"]
pay = s[(s["kind"] == "spending") & ~s["category"].isin(skip)].copy()
p = pay["Details"].str.replace(r"^(customer transfer to - |buy goods - |pay bill to |transfer to savings - )", "",
                               case=False, regex=True)
pay["payee"] = p.str.replace(r"^\d+\*{3}\d+ ", "", regex=True).str.replace(r"^\d{6} - ", "", regex=True).str.title()

snapshot = s["time"].max().normalize() + pd.Timedelta(days=1)
rfm = pay.groupby("payee").agg(last=("time", "max"), frequency=("amount", "count"), monetary=("amount", "sum"))
rfm["recency_days"] = (snapshot - rfm["last"].dt.normalize()).dt.days
rank = lambda v: v.rank(method="first")
rfm["R"] = pd.qcut(rank(-rfm["recency_days"]), 4, labels=[1, 2, 3, 4]).astype(int)   # 4 = most recent
rfm["F"] = pd.qcut(rank(rfm["frequency"]), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["M"] = pd.qcut(rank(rfm["monetary"]), 4, labels=[1, 2, 3, 4]).astype(int)
rfm["segment"] = np.select(
    [(rfm.R >= 3) & (rfm.F >= 3) & (rfm.M >= 3), (rfm.F >= 3) & (rfm.R >= 3),
     (rfm.M >= 3) & (rfm.R <= 2), (rfm.F >= 3) & (rfm.R <= 2)],
    ["Core partners", "Regular, smaller", "Big but quiet", "Fading regulars"], default="Occasional")

prof = rfm.groupby("segment").agg(payees=("monetary", "count"), median_recency_days=("recency_days", "median"),
                                  median_payments=("frequency", "median"), total_ksh=("monetary", "sum"))
prof["share_of_money_pct"] = (100 * prof["total_ksh"] / prof["total_ksh"].sum()).round(1)
print("payees scored:", len(rfm))
prof.sort_values("total_ksh", ascending=False).round(0)

## Step 5 · Which payments look unusual?

**What:** a **z-score** measures how far a payment is from the usual amount *for its category*, in standard deviations. We compare three rules and score each against the answer key.
**Why:** a fraud or review queue must catch the real problems without burying staff in false alarms. Amounts are skewed, so the log scale works better, and a Ksh 10,000 materiality floor removes small noise.

In [ ]:
out = s[(s["kind"] == "spending") & (s["category"] != "Fees")].copy()
g = out.groupby("category")["amount"]
out["z_raw"] = (out["amount"] - g.transform("mean")) / g.transform("std")
log_amt = np.log(out["amount"])
lg = log_amt.groupby(out["category"])
out["z_log"] = (log_amt - lg.transform("mean")) / lg.transform("std")
out = out.merge(answer_key[["Receipt No", "gen_anomaly"]], on="Receipt No", how="left")
planted = int(out["gen_anomaly"].sum())

def score(flag, name):
    caught = int((flag & (out["gen_anomaly"] == 1)).sum())
    print(f"{name:<28} flagged {int(flag.sum()):>3} | planted caught {caught}/{planted} | false alarms {int(flag.sum()) - caught}")

score(out["z_raw"].abs() > 3, "raw amount, |z| > 3")
score(out["z_log"].abs() > 3, "log amount, |z| > 3")
review = (out["z_log"].abs() > 3) & (out["amount"] > 10000)
score(review, "log z and amount > Ksh 10k")

In [ ]:
fig, ax = plt.subplots()
ax.scatter(out.loc[~review, "time"], out.loc[~review, "amount"], s=6, color=GREY, label="Within normal range")
ax.scatter(out.loc[review, "time"], out.loc[review, "amount"], s=28, color=AMBER, label="Flagged for review")
ax.set_yscale("log"); ax.set_title("Spending payments over time (log scale), flagged payments in amber")
ax.set_ylabel("Amount (Ksh, log scale)"); ax.legend(loc="lower right")
plt.tight_layout(); plt.show()

## Step 6 · Forecast next quarter's spending

**What:** fit a straight-line trend to monthly spending (leaving out December, which is always higher, and the flagged payments). Before trusting it, **back-test** it: fit on the first 18 months, predict the last 6, and compare with a naive "average of the last three months" forecast.
**Why:** a forecast is only useful if it beats the simple guess a person would make anyway.

In [ ]:
set_aside = out[review].groupby("month")["amount"].sum().reindex(flow.index, fill_value=0)
y = (flow["spending"] - set_aside).to_numpy()
xm = np.arange(len(y)); dec = (flow.index.month == 12)
slope, intercept = np.polyfit(xm[~dec], y[~dec], 1)
fit = slope * xm + intercept
uplift = float((y[dec] / fit[dec]).mean())
sd = (y - fit)[~dec].std(ddof=2)
print(f"trend: +Ksh {slope:,.0f} per month | December runs {uplift:.2f}x the trend")

k = 18   # back-test: train on the first 18 months, test on the last 6
b_slope, b_int = np.polyfit(xm[:k][~dec[:k]], y[:k][~dec[:k]], 1)
b_up = float((y[:k][dec[:k]] / (b_slope * xm[:k][dec[:k]] + b_int)).mean())
pred = (b_slope * xm[k:] + b_int) * np.where(dec[k:], b_up, 1.0)
naive_pred = np.full(len(y) - k, y[k - 3:k].mean())
mae = lambda a, p: np.abs(a - p).mean()
print(f"back-test average error: trend Ksh {mae(y[k:], pred):,.0f} | naive Ksh {mae(y[k:], naive_pred):,.0f}")

future = pd.period_range(flow.index[-1] + 1, periods=3, freq="M")
fc = slope * np.arange(len(y), len(y) + 3) + intercept
pd.DataFrame({"forecast_ksh": fc, "low_ksh": fc - 2 * sd, "high_ksh": fc + 2 * sd}, index=future.astype(str)).round(-2)

In [ ]:
fig, ax = plt.subplots()
ax.plot(flow.index.to_timestamp(), y / 1000, color=BLUE, marker="o", ms=4, label="Actual spending")
ax.plot(flow.index.to_timestamp(), fit / 1000, color=DARK, ls="--", label="Trend (fitted without December)")
xf = future.to_timestamp()
ax.plot(xf, fc / 1000, color=AMBER, marker="o", lw=2.2, label="Forecast")
ax.fill_between(xf, (fc - 2 * sd) / 1000, (fc + 2 * sd) / 1000, color=AMBER, alpha=0.2, label="Range (±2 sd)")
ax.set_title("Monthly spending, trend and three-month forecast"); ax.set_ylabel("Ksh thousand"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

## Step 7 · Final checks

In [ ]:
checks = {
    "raw = clean + rejected + duplicates": len(stmt) == len(s) + len(rejects) + n_dups,
    "receipt numbers unique": bool(s["Receipt No"].is_unique),
    "every row has a category": bool((s["category"] != "Other").all()),
    "monthly totals equal row totals": abs(flow[["income", "spending", "saving"]].sum().sum() - s["amount"].sum()) < 1,
}
for name, ok in checks.items():
    print("PASS" if ok else "FAIL", "-", name)

## Key findings and recommendations

- **Savings rate of about 26%** over 24 months, but some months spend more than comes in. *Automate a savings transfer on salary day.*
- **Cash withdrawals are the largest category (about a quarter of spending)**, so a large share of spending is invisible. *Tag cash and person-to-person transfers for one month.*
- **December runs about 1.65 times the trend.** *Budget for it and set money aside in October and November.*
- **All 12 planted anomalies caught with no false alarms** by the log z-score plus a Ksh 10,000 floor. *Send such payments to a confirm-or-dispute list.*
- The trend forecast beats the naive guess on held-out months, so it is fit for budgeting.

**Limitations.** The statement is synthetic and template-driven. Real statements contain other people's names and numbers, which must be masked before sharing.